In [49]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=400)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [50]:
tool.name

'wikipedia'

In [51]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
vector_db = FAISS.from_documents(documents, OllamaEmbeddings(model="llama3"))
retriever = vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12d420e00>)

In [52]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever, "langsmith-search",
      "Search any information about LangSmith.")

In [53]:
retriever_tool.name

'langsmith-search'

In [54]:
#Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=400)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name

'arxiv'

In [59]:
tools = [wiki_tool, arxiv_tool, retriever_tool]

In [60]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/rishaan/miniforge3/envs/langchain/lib/python3.12/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=400)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=400, arxiv_result=<class 'arxiv.Result'>)),
 Tool(name='langsmith-search', description='Search any information about LangSmith.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x11e7e36a0>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.fa

In [72]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1", temperature=0,verbose=False)

In [81]:
from langchain import hub
prompt = hub.pull("hwchase17/react")
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [86]:
#Agents
from langchain.agents import create_react_agent, AgentExecutor
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True,max_iterations=5,  # Limit iterations
    max_execution_time=60,  early_stopping_method="force")

In [87]:
agent_executor.invoke({"input": "Tell me about Artificial Intelligence"})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: This is a general knowledge question, so I should try to find information on Wikipedia.
Action: wikipedia
Action Input: search query: "Artificial Intelligence"Page: Content-based image retrieval
Summary: Content-based image retrieval, also known as query by image content (QBIC) and content-based visual information retrieval (CBVIR), is the application of computer vision techniques to the image retrieval problem, that is, the problem of searching for digital images in large databases (see this survey for a scientific overview of the CBIR field). Content-Here's my response:

Question: Tell me about Artificial Intelligence
Thought: This is a general knowledge question, so I should try to find information on Wikipedia.
Action: wikipedia
Action Input: search query: "Artificial Intelligence"Page: Content-based image retrieval
Summary: Content-based image retrieval, also known as query by image content (QBIC) and content-based visual information retrieval (CBVIR), is the application 

{'input': 'Tell me about Artificial Intelligence',
 'output': 'Agent stopped due to iteration limit or time limit.'}